# Rumo ao Desconhecido: Tratando Drift em ML
## Parte II — Dados reais: Bosch Production Line Performance

**Python Brasil 2026 · Blocos II-B a VI · ~110 min**

---

No notebook 01 **nós plantamos o drift**. Sabíamos o dia, a célula, o mecanismo
físico. Foi assim que provamos que o detector funciona — e é a única forma de
provar isso.

Agora o jogo muda:

- ninguém nos conta quando houve drift;
- as features são **anônimas** — `L3_S36_F3939` é a feature 3939, medida na
  estação 36 da linha 3, e nada mais;
- **0,58%** das peças falham (contra ~29% no sintético);
- cada peça percorre **uma rota diferente** pela fábrica → a maioria das
  colunas é NaN;
- **não existe coluna "dia"**: o tempo tem que ser *construído*.

> ## A pergunta que organiza este notebook
> Parte I: *"meu detector acerta?"*
> Parte II: **"o que eu faço quando ninguém me diz a resposta?"**

**Entregável:** um eixo temporal derivado dos timestamps, um monitor segmentado
por estação, detecção de *presence drift*, métricas que sobrevivem ao
desbalanceamento extremo, e um **pipeline com exit code** que sabe a diferença
entre *"o modelo envelheceu"* e *"o sensor mentiu"*.

In [ ]:
# ============================================================
#  CÉLULA 0 — SETUP (idempotente)
# ============================================================
!curl -sSL https://raw.githubusercontent.com/arcursino/python-br-2026/main/setup_colab.py -o /tmp/setup_colab.py
%run /tmp/setup_colab.py

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from driftkit.notebook import banner, requer, checkpoint, aposte, resposta, cli
from driftkit.data import dir_dados, baixar_bosch, carregar_meta_bosch, carregar_features_bosch

DATA = dir_dados()
plt.rcParams.update({
    "font.size": 14, "axes.titlesize": 18, "axes.labelsize": 15,
    "legend.fontsize": 13, "xtick.labelsize": 13, "ytick.labelsize": 13,
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.25,
})
pd.set_option("display.width", 180)

---
## A ponte: herdando o contrato v1

O que atravessa da Parte I é o **método**. Os **parâmetros** serão remedidos
aqui — e o que acontece quando você tenta transportar o número é o resultado
mais contraintuitivo deste notebook.

> Se você fechou o Colab entre os blocos, relaxe: existe fallback. Você será
> avisado de que está usando os limiares da minha execução, não da sua.

In [ ]:
from driftkit.state import carregar_contrato

CFG_V1 = carregar_contrato()

PISO_SINTETICO = CFG_V1["limiares"]["psi_piso_aa"]
PSI_ALARME_SINT = CFG_V1["limiares"]["psi_alarme"]

print("\n--- herdado SEM alteração (é MÉTODO, não parâmetro) ---")
for k, v in CFG_V1["guardas_retreino"].items():
    print(f"  guarda      {k:<18} = {v}")
for k, v in CFG_V1["assinaturas_diagnosticas"].items():
    print(f"  assinatura  {k:<18} = {v}")

print("\n--- herdado SOB SUSPEITA (calibrado em outro regime) ---")
fx = CFG_V1["fixture"]
print(f"  psi_piso_aa = {PISO_SINTETICO:.5f}")
print(f"  psi_alarme  = {PSI_ALARME_SINT:.5f}")
print(f"    medidos com {fx['features_monitoradas']} features, "
      f"{fx['amostras_por_janela']:,} amostras/janela, prevalência {fx['prevalencia']:.1%}")
print("\n>>> Aqui teremos ~160 features, ~40.000 amostras/janela, prevalência 0.58%.")
print(">>> Estes dois números SERÃO recalibrados. O método é que transfere.")

---
# 🏭 Bloco II-B — Construindo o eixo temporal (20 min)

## Por que não dá para fazer `pd.read_csv(train_numeric.csv)`

`1.183.747 linhas × 969 features × 4 bytes` (float32) ≈ **4,6 GB** só do
numérico, antes de qualquer cópia intermediária do pandas. Com float64, o dobro.

### A estratégia em três passos — a resposta certa em entrevista de engenharia de dados

1. **Passo barato primeiro.** `Id` + `Response` custam ~10 MB. Planeje com isso.
2. **Reduza por chunks, não carregue por chunks.** Para o eixo temporal
   precisamos de *um número por peça* (o mínimo dos timestamps). Isso é uma
   **redução**: processe 100k linhas, guarde o agregado, descarte o chunk.
   Pico de memória ~600 MB em vez de 5 GB.
3. **Selecione antes de carregar.** Duas passagens (perfil → carga) custam menos
   que uma carga inteira.

> ### 📌 Nós NÃO vamos rodar isso agora
>
> O código completo está em **`scripts/preprocess_bosch.py`** — leia, é curto e
> comentado. Eu rodei em casa (~20 min) e publiquei os dois `.parquet` no GitHub
> Releases. Você acabou de baixá-los no intervalo.
>
> **Vamos projetar e discutir o código**, porque o padrão vale mais que o
> resultado. Mas 25 minutos de I/O em sala é tempo que você não recupera.

In [ ]:
# leia o coração do script — a redução por chunks com índices precomputados
!sed -n '/def construir_meta/,/^    return meta/p' $RAIZ_REPO/scripts/preprocess_bosch.py | head -80

In [ ]:
baixar_bosch()  # se você não rodou no intervalo

meta = carregar_meta_bosch()
print(f"\nmeta: {len(meta):,} peças × {meta.shape[1]} colunas")
print(f"taxa de falha: {meta.Response.mean():.4%}  ({meta.Response.sum():,} falhas)")
print(f"semanas       : {meta.semana.min()} → {meta.semana.max()}")
meta[["Id", "t_min", "t_max", "duracao", "n_estacoes", "semana", "Response"]].head()

### ⚠️ Três armadilhas específicas do Bosch — antes de qualquer análise

**(a) O vazamento do `Id`.** Existe correlação conhecida entre a ordem do `Id` e
a `Response` nesta competição. Isso rendeu ótimos scores de leaderboard e
**nenhum aprendizado sobre a fábrica**. Se você construir o eixo temporal a
partir do `Id`, está monitorando o vazamento, não o processo. **Aqui o tempo vem
exclusivamente das colunas `_D`.**

**(b) Features derivadas de vizinhos.** "Carga da estação" ou "peça anterior"
são legítimas em produção, mas precisam ser calculadas **causalmente** (só com o
passado). Um `groupby` ingênuo sobre o dataset inteiro contamina a referência.

**(c) Ausência de ground truth.** Nunca poderemos afirmar *"houve drift na
semana 14"*. Podemos afirmar: *"o monitor cruzou o limite de controle na semana
14, a estação L3_S32 é a maior contribuinte, e a hipótese a investigar é X"*.
**Essa é a redação correta de um alerta.**

In [ ]:
# ============================================================
#  O eixo temporal existe? Vamos olhar o volume e a taxa de falha.
# ============================================================
requer("meta")
por_semana = meta.groupby("semana", observed=True).agg(
    n=("Id", "size"), falha=("Response", "mean"), estacoes=("n_estacoes", "mean")
)

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
axes[0].bar(por_semana.index, por_semana.n, color="tab:blue", alpha=0.8)
axes[0].set_ylabel("peças"); axes[0].set_title("O eixo temporal que construímos")
axes[1].plot(por_semana.index, por_semana.falha * 100, "o-", color="tab:red", lw=2.5)
axes[1].axhline(meta.Response.mean() * 100, ls="--", color="gray",
                label=f"média global {meta.Response.mean():.3%}")
axes[1].set_ylabel("falha (%)"); axes[1].legend()
axes[2].plot(por_semana.index, por_semana.estacoes, "s-", color="tab:green", lw=2.5)
axes[2].set_ylabel("estações/peça"); axes[2].set_xlabel("semana")
plt.tight_layout(); plt.show()

print(por_semana.round(4).to_string())
print("\n>>> As primeiras e últimas semanas têm volume baixo (bordas do recorte).")
print(">>> Elas NÃO servem como referência nem como janela. Vamos excluí-las —")
print(">>> e essa decisão precisa ser explícita, não implícita.")

In [ ]:
# ============================================================
#  PRESENCE DRIFT — o sinal mais barato deste dataset
#  Ausência aqui não é bug: é a ROTA de produção.
#  E rotas mudam, sem que nenhuma distribuição de valores se mexa.
# ============================================================
banner("Presence drift", "π_s(t) = P(peça visita estação s | semana t)", icone="🔀")

vis_cols = [c for c in meta.columns if c.startswith("vis_")]
pi = meta.groupby("semana", observed=True)[vis_cols].mean()

# quais estações mais mudaram de presença ao longo do tempo?
variacao = (pi.max() - pi.min()).sort_values(ascending=False)
print("estações com maior variação de presença:")
print(variacao.head(8).round(3).to_string())

fig, ax = plt.subplots(figsize=(13, 5))
for c in variacao.head(5).index:
    ax.plot(pi.index, pi[c], lw=2.5, marker="o", ms=4, label=c.replace("vis_", ""))
ax.set_xlabel("semana"); ax.set_ylabel("π_s(t)")
ax.set_title("Roteamento muda — e KS, χ² e PSI são cegos a isso")
ax.legend(ncol=5, loc="lower left")
plt.tight_layout(); plt.show()

print(
    "\n>>> Quando a fábrica desvia 30% do volume da estação 24 para a 25,\n"
    ">>> NENHUMA distribuição de valores se mexe — mas o significado de\n"
    ">>> 'peça típica' muda. Monitorar π_s(t) custa quase nada.\n"
    ">>> Se você só monitora valores observados, você é cego a isso."
)

---
# 📊 Bloco III — Data drift em dado real (40 min)

## III.0 Recalibração: o momento da verdade

Agora fazemos com o Bosch exatamente o que fizemos com a fixture: medimos o
piso de ruído **deste** detector, **neste** regime.

Mas com duas diferenças em relação ao notebook 01 — e as duas importam:

**1. Existe um valor esperado analítico.** O PSI é a divergência de Jeffreys, e
sob a hipótese nula (duas amostras da mesma distribuição) ele é um χ² escalado:

$$\mathrm{E}[\text{PSI}] = (B-1)\left(\frac{1}{n}+\frac{1}{m}\right)
\qquad p_{95} = \chi^2_{0,95}(B-1)\left(\frac{1}{n}+\frac{1}{m}\right)$$

Isso muda tudo: o piso deixa de ser um número medido no escuro e passa a ser um
número medido **contra uma previsão**. E a razão entre os dois é que carrega a
informação.

**2. O split em bloco.** Vamos dividir a referência em blocos **consecutivos no
tempo**, não aleatoriamente — porque o split aleatório é provadamente cego a
contaminação da própria janela de referência. Mais sobre isso em duas células.

In [ ]:
aposte(
    "No sintético, o piso de ruído do detector foi ~0.017 com 9 features e 2.700 "
    "amostras por janela. Aqui teremos 160 features — 18x mais — mas também "
    "40.000 amostras por janela. O piso vai ser MAIOR ou MENOR que o do sintético?",
    ("maior", "menor", "parecido"),
)

In [ ]:
from driftkit.detectors import DriftDetector

feats = carregar_features_bosch()
FEATS_BOSCH = [c for c in feats.columns if c not in ("Id", "Response")]
print(f"features: {len(FEATS_BOSCH)} | {feats.shape[0]:,} peças")

# ── junta o eixo temporal às features ──────────────────────────────────────
# `t_min` ENTRA no merge: é ele que define a ordem temporal fina dentro de
# cada semana, e sem ela o split em bloco não tem sentido.
# (Ele não entra em FEATS_BOSCH, então o detector o ignora.)
base = meta[["Id", "semana", "t_min", "n_estacoes", "duracao", "Response"]].merge(
    feats.drop(columns=["Response"]), on="Id", how="inner", validate="one_to_one"
)

# semanas de borda fora — decisão EXPLÍCITA
vol = base.groupby("semana", observed=True).size()
semanas_ok = vol[vol > vol.median() * 0.4].index.tolist()

# ── ORDENAÇÃO TEMPORAL — não é cosmética ───────────────────────────────────
# Ordenamos `base` UMA vez. Assim toda janela `base[base.semana == s]` herda a
# ordem temporal, e o split em bloco funciona em qualquer uma delas.
base = base[base.semana.isin(semanas_ok)].sort_values("t_min", kind="stable")

SEM_REF = semanas_ok[:4]
ref_bosch = base[base.semana.isin(SEM_REF)]   # já sai ordenada de `base`

print(f"\nreferência: semanas {SEM_REF} → {len(ref_bosch):,} peças")
print(f"janelas    : semanas {semanas_ok[4]} a {semanas_ok[-1]}")
print(f"ordenada por t_min? {ref_bosch.t_min.is_monotonic_increasing}")

det_bosch = DriftDetector.from_reference(
    ref_bosch, num=FEATS_BOSCH, cat=[],
    psi_limiar=PSI_ALARME_SINT,   # ⚠️ herdado do sintético — de propósito
    alpha=0.05, min_obs=2000,
)

### 🔎 Por que o split tem que ser no tempo, e não aleatório

O teste A/A clássico permuta a referência e divide ao meio. Isso mede o ruído do
instrumento — e é útil. Mas ele **não pode** detectar o problema mais grave que
uma referência pode ter: conter mais de um regime.

> Se a sua referência é uma mistura de dois regimes, a permutação distribui os
> dois **igualmente** entre as metades. As duas metades passam a ser amostras da
> **mesma** mistura — e mistura embaralhada é permutacionalmente trocável.
> O teste dá verde. Sempre.

Medido em simulação (contaminação por um 2º regime a +1σ):

| contaminação | split aleatório | split em bloco |
|---:|---:|---:|
| 0% | 1,0 | 0,6 |
| 5% | 1,0 | 0,8 |
| 10% | 1,0 | **2,8** |
| 20% | 1,0 | **9,6** |
| 50% | 1,0 | **55,8** |

O split em bloco compara o passado com o passado-mais-recente. Aí a
heterogeneidade aparece. É por isso que `calibrar_piso` roda os **dois** modos:
o aleatório mede o instrumento, o bloco diagnostica o gabarito.

In [ ]:
# ============================================================
#  RECALIBRAÇÃO no dado real — piso + diagnóstico da referência
# ============================================================
from driftkit.detectors import piso_analitico, n_equivalente
from driftkit.state import comparar_regimes

requer("det_bosch", "ref_bosch", "CFG_V1")

# ── o contrato v1 no formato de calibração ─────────────────────────────────
# Se o notebook 01 gravou `fixture.calibracao`, usamos direto. Senão,
# reconstruímos a partir dos limiares — dá no mesmo.
_fx = CFG_V1["fixture"]
if "calibracao" in _fx:
    CAL_V1 = _fx["calibracao"]
else:
    _n1 = _fx.get("n_referencia", _fx["amostras_por_janela"] * 10)
    _k1 = len(CFG_V1["features"]["num"])
    _b1 = CFG_V1["limiares"].get("bins", 10)
    CAL_V1 = {
        "n_referencia": _n1, "k_features_num": _k1, "bins": _b1,
        "piso_aa": PISO_SINTETICO,
        "psi_alarme_sugerido": PSI_ALARME_SINT,
        "piso_analitico": piso_analitico(_n1 // 2, _n1 // 2, bins=_b1, k_features=_k1),
        "H": float("nan"), "referencia": "?",
    }

# ── calibração: os dois modos de uma vez ──────────────────────────────────
cal_v2 = det_bosch.calibrar_piso(modo="ambos", n_repeticoes=20, n_blocos=4, seed=7)
for k, v in cal_v2.items():
    print(f"  {k:<26} {v}")

In [ ]:
# ============================================================
#  O piso que REALMENTE importa: referência × janela operacional
# ============================================================
# O teste A/A divide a referência ao meio: n/2 contra n/2.
# Mas em OPERAÇÃO você compara a referência INTEIRA contra uma janela
# pequena — e o piso escala com (1/n_ref + 1/n_janela), que é maior.
#
# Calibrar no split 50/50 e operar com janela pequena SUBESTIMA o ruído,
# e produz alarme falso sistemático. Esta célula corrige isso.
requer("cal_v2")

n_ref = len(ref_bosch)
n_jan = int(base.groupby("semana", observed=True).size().median())

piso_op = piso_analitico(n_ref, n_jan, bins=det_bosch.bins,
                         k_features=len(FEATS_BOSCH))

# a razão medido/previsto corrige marginais degeneradas (empates, zeros
# inflados) — no Bosch ela costuma ficar abaixo de 1, porque bins colapsam
razao = cal_v2.get("razao_aa", 1.0)
razao = razao if (razao is not None and np.isfinite(razao) and razao > 0) else 1.0

PISO_REAL = piso_op * razao
PSI_ALARME_REAL = 3.0 * PISO_REAL

print(f"  piso do A/A (n/2 vs n/2)     {cal_v2['piso_aa']:.6f}")
print(f"  piso analítico previsto      {cal_v2['piso_analitico']:.6f}"
      f"   (razão medido/previsto {razao:.2f})")
print(f"  piso OPERACIONAL             {PISO_REAL:.6f}"
      f"   (ref={n_ref:,} × janela={n_jan:,})")
print(f"  → limiar recalibrado         {PSI_ALARME_REAL:.6f}   (3× o piso)")

print(
    "\n  A razão medido/previsto é o seu TESTE DE SANIDADE do código:\n"
    "  se ela ficar muito longe de 1 em dado bem-comportado, o bug está\n"
    "  na sua implementação de PSI — não na teoria."
)

In [ ]:
# ============================================================
#  O DIAGNÓSTICO: a janela de referência é homogênea?
# ============================================================
#   H = piso medido no bloco / piso analítico previsto
#
#   H < 2   → referência homogênea no eixo temporal
#   2 ≤ H < 5 → suspeita: investigue antes de confiar no limiar
#   H ≥ 5   → CONTAMINADA: mais de um regime dentro da referência.
#             Nenhum limiar te salva disso.
requer("cal_v2")

H = cal_v2.get("H", float("nan"))
H = H if H is not None else float("nan")

print("=" * 68)
if np.isfinite(H):
    print(f"  ÍNDICE H = {H:.2f}   →   referência "
          f"{str(cal_v2.get('referencia', '?')).upper()}")
else:
    print("  ÍNDICE H não estimável (blocos pequenos demais)")
print("=" * 68)

if np.isfinite(H) and H >= 5:
    print(f"\n  ⚠️  o bloco {cal_v2.get('bloco_mais_divergente')} é o mais divergente.")
    print("      Há mais de um regime dentro da referência: uma campanha, uma")

    print("      parada, uma troca de fornecedor que ninguém anotou.")
    print("      O limiar calculado acima é aritmeticamente correto e")
    print("      praticamente INÚTIL — encurte ou segmente a referência.")
    print("\n      >>> O problema não é o limiar. É o GABARITO.")
elif np.isfinite(H) and H >= 2:
    print("\n  ⚠️  suspeita de heterogeneidade. Vale olhar o volume e o mix")
    print("      por semana dentro da referência antes de confiar no limiar.")
else:
    print("\n  ✅ a referência é internamente homogênea no eixo temporal.")
    print("     O limiar calibrado acima é confiável.")

In [ ]:
# ============================================================
#  E se você simplesmente TRANSPORTASSE o limiar do sintético?
#  (o resultado é o oposto do que quase todo mundo espera)
# ============================================================
requer("CAL_V1", "cal_v2", "piso_op")

cmp = comparar_regimes(CAL_V1, {**cal_v2, "piso_analitico": piso_op},
                       nome_v1="sintético", nome_v2="Bosch")

print("=" * 72)
print(f"  limiar do sintético = {cmp['limiar_v1_em_pisos_de_v2']}× o piso do Bosch")
print("=" * 72)
print(f"\n  {cmp['veredicto']}")

print(f"\n  e o folclórico 0.1 corresponde a uma janela de "
      f"~{n_equivalente(0.10, bins=det_bosch.bins):.0f} observações.")
print(f"  a sua janela tem {n_jan:,}.")
print(f"  → o 0.1 está {0.10 / piso_op:.0f}× solto aqui.")

print(
    "\n" + "─" * 72 + "\n"
    "  Repare que o limiar herdado NÃO ficou ruidoso. Ficou CEGO.\n"
    "\n"
    "  O piso de ruído cai com o tamanho da janela — piso ∝ (1/n + 1/m).\n"
    "  A janela do Bosch é ~15× a da fixture, e esse ganho de n domina\n"
    "  folgadamente o efeito de monitorar 160 features em vez de 9.\n"
    "\n"
    "  >>> Um monitor com limiar FIXO fica progressivamente mais cego\n"
    "  >>> conforme o seu volume de dados cresce. Você paga por mais\n"
    "  >>> dados e joga fora todo o poder estatístico que eles compram.\n"
    + "─" * 72
)

det_bosch = det_bosch.com(psi_limiar=PSI_ALARME_REAL)

---
## 🛠️ VOCÊ IMPLEMENTA #3 — limiar por estrato de cobertura (6 min)

As 160 features não são iguais. Algumas estão presentes em 95% das peças; outras
em 10% (a peça não passou por aquela estação). Uma feature com 10% de cobertura
tem **10× menos amostras por janela** — e você já sabe, pela fórmula do piso, o
que isso faz com o ruído: ele cresce com $$1/\sqrt{n}$$.

Usar o mesmo limiar para todas produz alarme falso sistemático justamente nas
features mais frágeis.

Escreva `limiar_de(cobertura, piso_base)`: quanto **menor** a cobertura, **maior**
o limiar.

In [ ]:
def limiar_de(cobertura: float, piso_base: float) -> float:
    """Limiar ajustado pela cobertura da feature.

    O erro padrão de um estimador escala com 1/sqrt(n). Se a cobertura cai
    por um fator k, o n cai por k, e o ruído cresce por sqrt(k).

    Requisitos: sempre positivo; monotonicamente decrescente na cobertura.
    """
    # TODO: uma ou duas linhas. Dica: piso_base / sqrt(cobertura)
    raise NotImplementedError

In [ ]:
from driftkit.testing import checar_limiar_por_estrato

checar_limiar_por_estrato(limiar_de)

In [ ]:
# ============================================================
#  O monitor no tempo — e a redação CORRETA de um alerta
# ============================================================
requer("det_bosch", "base")

trilha_bosch = []
for sem in semanas_ok[4:]:
    w = base[base.semana == sem]
    rel = det_bosch.report(w)
    trilha_bosch.append({
        "semana": sem, "n": len(w),
        "n_drift": rel.n_drift,
        "efeito_max": rel.efeito_max,
        "nao_mensuravel": rel.n_nao_mensuravel,
        "top": ", ".join(rel.top_features[:2]),
    })
trilha_bosch = pd.DataFrame(trilha_bosch)
print(trilha_bosch.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(trilha_bosch.semana, trilha_bosch.n_drift, color="tab:orange", alpha=0.85)
ax.set_xlabel("semana"); ax.set_ylabel("features em drift (de 160)")
ax.set_title("Monitor de data drift — Bosch, limiar recalibrado")
ax2 = ax.twinx(); ax2.grid(False)
ax2.plot(trilha_bosch.semana, trilha_bosch.efeito_max, "k-o", lw=2.5, label="efeito máx")
ax2.axhline(det_bosch.psi_limiar, color="red", ls="--", lw=2,
            label=f"limiar recalibrado ({det_bosch.psi_limiar:.4f})")
ax2.axhline(PSI_ALARME_SINT, color="tab:blue", ls="-.", lw=2,
            label=f"limiar do sintético ({PSI_ALARME_SINT:.3f}) — CEGO aqui")
ax2.axhline(0.10, color="gray", ls=":", lw=2, label="o 0.1 do folclore")
ax2.legend(loc="upper right", fontsize=11); ax2.set_ylabel("efeito (PSI)")
plt.tight_layout(); plt.show()

pior = trilha_bosch.loc[trilha_bosch.efeito_max.idxmax()]
print(
    f"\n❌ ERRADO: 'houve drift na semana {pior.semana:.0f}.'\n"
    f"✅ CERTO : 'o monitor cruzou o limite de controle na semana {pior.semana:.0f} "
    f"(efeito {pior.efeito_max:.4f}, {pior.n_drift:.0f} features).\n"
    f"           Maiores contribuintes: {pior.top}.\n"
    f"           {pior.nao_mensuravel:.0f} features não mensuráveis nesta janela.\n"
    f"           Hipótese a investigar: mudança de setup na estação correspondente.'"
)

---
## 🗣 Atividade 2 — Interrogue o seu próprio detector (5 min, em trios)

Olhem o gráfico acima. Se o monitor acusa drift em **quase todo** o período,
há quatro hipóteses possíveis — e a ordem de investigação importa.

**Em trios, 4 minutos:** quais são as quatro? Como vocês testariam cada uma?

*Dica: sempre suspeite do seu instrumento antes de suspeitar do mundo. E note
que uma delas você já sabe medir — é o índice H.*

## ☕ Intervalo (15 min)

---
# 🎯 Bloco IV — Concept drift sob desbalanceamento extremo (30 min)

## IV.1 Por que 0,58% quebra o instrumental da Parte I

Com prevalência $$\pi \approx 0{,}0058$$, um modelo que prevê "nunca falha"
acerta **99,42%**. Consequências, todas práticas:

1. **Acurácia é inútil.** Ignore.
2. **F1 e MCC dependem violentamente do limiar.** Reportar F1@0.5 aqui não
   significa nada.
3. **ROC-AUC é otimista e insensível** — a região que importa (topo do ranking)
   é uma fração minúscula da curva. Use **PR-AUC** como métrica de *ranking* e
   **MCC** no melhor limiar como métrica de *decisão* (foi a métrica oficial da
   competição).
4. **DDM/EDDM sobre o erro binário param de funcionar.** Eles monitoram a taxa
   de erro; aqui ela é ~0,6% e dominada pelo prior. Um colapso de performance
   mexe a terceira casa decimal.

> ### 🗣 Nota de honestidade — a ementa prometia F1 e RMSE
>
> RMSE não se aplica a classificação. F1@0.5 com 0,58% de prevalência é ruído.
> Vou entregar **MCC e PR-AUC** no lugar — e a célula abaixo mostra, em 20
> segundos, exatamente por que. As métricas prometidas estão implementadas em
> `modelo.metricas_completas()` para você confrontar.

In [ ]:
from driftkit.modelo import treinar, metricas, metricas_completas, melhor_limiar_mcc

modelo_b = treinar(ref_bosch, num=FEATS_BOSCH, cat=[], alvo="Response")

w_teste = base[base.semana == semanas_ok[8]]
comp = metricas_completas(modelo_b, w_teste, num=FEATS_BOSCH, cat=[], alvo="Response")

print("CONFRONTO DE MÉTRICAS — mesma janela, mesmo modelo\n" + "-" * 52)
for k, v in comp.items():
    print(f"  {k:<24} {v}")
print("-" * 52)
print(
    f"\n>>> O modelo acerta {comp['acuracia_do_modelo']:.1%}.\n"
    f">>> Um `return 0` acerta {comp['acuracia_do_burro']:.1%}.\n"
    f">>> F1@0.5 = {comp['f1_at_05']}. MCC no limiar ótimo = {comp['mcc_otimo']}.\n"
    f">>> Sempre reporte o par (limiar, métrica). Métrica de decisão sem o\n"
    f">>> limiar que a produziu não é reproduzível."
)

In [ ]:
# ============================================================
#  A trilha de performance — camada 3, com métricas que resistem
# ============================================================
perf = []
for sem in semanas_ok[4:]:
    w = base[base.semana == sem]
    m = metricas(modelo_b, w, num=FEATS_BOSCH, cat=[], alvo="Response")
    perf.append({"semana": sem, **m.to_dict()})
perf = pd.DataFrame(perf)
print(perf.to_string(index=False))

n_nao = perf.pr_auc.isna().sum()
if n_nao:
    print(f"\n⚠️  {n_nao} janela(s) NÃO estimável(is) — poucos positivos.")
    print("    Repare que devolvemos None, não 0. `nan < 0.7` é False:")
    print("    janela não estimável passaria como SAUDÁVEL. É o pior modo de falha.")

fig, (a1, a2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
a1.plot(perf.semana, perf.pr_auc, "o-", lw=2.5, color="tab:blue", label="PR-AUC (ranking)")
a1.plot(perf.semana, perf.roc_auc, "s--", lw=2, color="tab:gray", label="ROC-AUC (otimista)")
a1.legend(); a1.set_ylabel("ranking")
a1.set_title("Camada 3 — e a diferença entre PR-AUC e ROC-AUC com 0.58%")
a2.plot(perf.semana, perf.mcc, "o-", lw=2.5, color="tab:red", label="MCC (decisão)")
a2.plot(perf.semana, perf.limiar, "^:", lw=2, color="tab:purple", label="limiar ótimo τ*")
a2.legend(); a2.set_ylabel("decisão"); a2.set_xlabel("semana")
plt.tight_layout(); plt.show()

## IV.2 O drift que quase todo mundo esquece: **o limiar**

Seu modelo produz um score $$\hat{p}$$. A decisão usa um limiar $$\tau$$. E:

$$\tau^* \text{ se desloca quando } P(Y) \text{ muda, mesmo com } P(Y\mid X) \text{ intacto.}$$

Ou seja: **o modelo pode estar perfeito e a decisão estar errada.** O MCC cai, o
alarme dispara, alguém pede retreino — e a correção certa custava *uma linha de
código*.

É o mesmo padrão do offset do transdutor na Parte I:
**a intervenção mais barata quase nunca é retreinar.**

In [ ]:
# quanto do 'colapso de MCC' é resolvido apenas movendo tau?
requer("perf", "modelo_b")

TAU_FIXO = perf.limiar.dropna().iloc[0]
cmp_tau = []
for sem in semanas_ok[4:]:
    w = base[base.semana == sem]
    m_fixo = metricas(modelo_b, w, num=FEATS_BOSCH, cat=[], alvo="Response", limiar=TAU_FIXO)
    m_otim = metricas(modelo_b, w, num=FEATS_BOSCH, cat=[], alvo="Response")
    cmp_tau.append({"semana": sem, "mcc_tau_fixo": m_fixo.mcc,
                    "mcc_tau_reajustado": m_otim.mcc, "tau_otimo": m_otim.limiar})
cmp_tau = pd.DataFrame(cmp_tau)
print(cmp_tau.to_string(index=False))

g = (cmp_tau.mcc_tau_reajustado - cmp_tau.mcc_tau_fixo).mean()
print(f"\n>>> Ganho médio de MCC apenas REAJUSTANDO o limiar: {g:+.4f}")
print(">>> Custo: uma linha de código. Custo de um retreino: uma GPU e um deploy.")
print(">>> Isto é o primeiro degrau da escada de intervenção.")

In [ ]:
# ============================================================
#  Detectores sequenciais — DEMO (5 min, não é hands-on)
#  A lição não é 'qual detector'. É QUAL SINAL você alimenta nele.
# ============================================================
from driftkit.modelo import logloss_por_amostra

try:
    from river.drift import ADWIN

    sinais = {
        "erro binário  (o jeito errado)": np.concatenate([
            (modelo_b.predict(base[base.semana == s][FEATS_BOSCH]) !=
             base[base.semana == s].Response.to_numpy()).astype(float)
            for s in semanas_ok[4:]]),
        "log-loss/amostra (o jeito certo)": np.concatenate([
            logloss_por_amostra(modelo_b, base[base.semana == s],
                                num=FEATS_BOSCH, cat=[], alvo="Response")
            for s in semanas_ok[4:]]),
    }

    for nome_sinal, sinal in sinais.items():
        det_seq = ADWIN(delta=0.002)
        mudancas = 0
        for x in sinal[::50]:
            det_seq.update(float(x))
            if getattr(det_seq, "drift_detected", False):
                mudancas += 1
        print(f"{nome_sinal:<34} → {mudancas} pontos de mudança detectados")

    print(
        "\n>>> Mesmo ADWIN. Sinal diferente, resultado diferente.\n"
        ">>> Com prevalência de 0.58%, o erro binário é ~0.6% e dominado pelo\n"
        ">>> prior: um colapso do modelo mexe a terceira casa decimal.\n"
        ">>> A solução não é trocar o detector. É trocar o SINAL DE ENTRADA\n"
        ">>> por algo contínuo e sensível."
    )
except ImportError:
    print("river indisponível. O argumento sobrevive: o problema é o SINAL,")
    print("não o detector. Código completo no repositório.")

---
# 🛡️ Bloco V — Mitigação: o pipeline que sabe se recusar (20 min)

## V.1 A escada de intervenção — do mais barato ao mais caro

| # | intervenção | custo | quando |
|---|---|---|---|
| 1 | **reajustar o limiar** $$\tau$$ | 1 linha | $$P(Y)$$ mudou, $$P(Y\mid X)$$ intacto |
| 2 | **recalibrar** probabilidades (isotônica/Platt) | minutos | scores descalibrados |
| 3 | **retreinar** com dado recente | GPU + deploy | o mundo mudou de verdade |
| 4 | **rearquitetar** features | semanas | mudança estrutural do processo |
| 5 | ⛔ **não fazer nada de ML** — chamar engenharia | ordem de serviço | causa física |

O degrau 5 não é desistência. É a resposta correta — e é o que nenhum pipeline
de MLOps convencional sabe fazer.

## V.2 A armadilha do retreino cego, em quatro atos

> 1. **SEDUÇÃO** — retreina no dado do sensor mentiroso. AUC vai de 0,53 a 0,85. ✅ LGTM, merge.
> 2. **ENCOBRIMENTO** — os alarmes somem. O dashboard fica verde.
> 3. **REALIDADE** — Cpk 0,35. 152.000 PPM fora de spec. A física não mudou.
> 4. **ARMADILHA** — quando a metrologia finalmente recalibrar, **o modelo quebra**.

> ### 🏭 Axioma industrial
> Retreinar um modelo sobre dado de sensor descalibrado é **institucionalizar o
> defeito mecânico dentro do código de IA**.

In [ ]:
# ============================================================
#  As 4 guardas clássicas — e a prova de que não bastam
#  Voltamos à fixture, porque só lá temos GABARITO da causa.
# ============================================================
from driftkit.fixtures import NUM, CAT, RANGES, gerar_fixture, janela
from driftkit.detectors import DriftDetector as DD
from driftkit.policy import PoliticaRetreino, tabela_das_quatro_guardas, Acao
from driftkit.modelo import metricas as met_sint

dfs = gerar_fixture(seed=7)
refs = dfs.query("dia < 30")
det_s = DD.from_reference(refs, num=list(NUM), cat=list(CAT), ranges=RANGES,
                          psi_limiar=PSI_ALARME_SINT)
mod_s = treinar(refs, num=NUM, cat=CAT)
pol = PoliticaRetreino(detector=det_s, piso_aa=PISO_SINTETICO, k_de_n=(1, 1))

w = janela(dfs, 110)                      # TC-3: o transdutor descalibrado
seg = w.query("equipamento == 'CEL-02'")
d = pol.avaliar(
    w, dia=110, segmento="CEL-02",
    auc_global=met_sint(mod_s, w, num=NUM, cat=CAT).roc_auc,
    auc_segmento=met_sint(mod_s, seg, num=NUM, cat=CAT).roc_auc,
)
print(d)
print()
print(tabela_das_quatro_guardas(d).to_string(index=False))
print(
    "\n>>> Leia a tabela devagar. AS QUATRO GUARDAS CLÁSSICAS PASSAM.\n"
    ">>> Elas foram desenhadas para impedir retreino DESNECESSÁRIO.\n"
    ">>> Nenhuma foi desenhada para impedir retreino DANOSO — porque danoso\n"
    ">>> e necessário são indistinguíveis por qualquer métrica de ML.\n"
    ">>> A distinção é FÍSICA. Por isso a guarda 5 é um BLOQUEIO, não um score."
)

---
## V.3 De opinião a infraestrutura: **o exit code é a política**

Enquanto a decisão de retreinar mora num `if` dentro de uma célula, ela é uma
opinião. Quando ela vira **código de saída de processo**, ela é infraestrutura:
GitHub Actions, Airflow, cron, Argo — qualquer orquestrador sabe ler um exit
code, e nenhum deles precisa saber o que é PSI.

| exit | significado | o que o orquestrador faz |
|:---:|---|---|
| `0` | sem drift acionável | nada |
| `10` | retreino **aprovado** | dispara o job de retreino |
| `20` | retreino **bloqueado** | abre ticket para engenharia de processo |
| `30` | dados insuficientes | registra — e **não** confunde com "tudo bem" |

In [ ]:
# O pipeline decidindo dia após dia — o 'cron' do tutorial.
# Repare na coluna `exit`: é isso que o orquestrador consome.
cli("simular", "--de", "35", "--ate", "130", "--passo", "5")

In [ ]:
# ⚠️ Por que não `!driftkit decide ...`?
# A magia `!` do Jupyter NÃO propaga o exit code — ela imprime o stdout e
# descarta o código de saída. E o exit code é onde mora a política inteira.
p_janela = DATA / "janela_tc3.parquet"
if not p_janela.exists():
    janela(dfs, 110).to_parquet(p_janela)

codigo = cli("decide", str(p_janela), "--dia", "110")

print(f"\n>>> exit={codigo}. Em produção, é este número que decide se o")
print(">>> GitHub Actions retreina ou abre uma issue para a metrologia.")

In [ ]:
# E o workflow que consome esse exit code — a 'esteira de MLOps' da ementa.
# Repare no penúltimo step: quando exit=20, ele NÃO retreina. Abre um ticket.
!sed -n '/exit 20/,/^      # ===/p' $RAIZ_REPO/.github/workflows/drift.yml | head -40

---
## 🛠️ VOCÊ IMPLEMENTA #5 — a guarda que **você** inventa (8 min, ⭐⭐⭐)

As cinco guardas cobrem: blip, sazonalidade, ruído, challenger fraco e causa
física. Falta uma.

**Desgaste de ferramenta** é gradual: não tem degrau, não viola range, e a
magnitude *por janela* fica **sempre abaixo** do limiar. Todas as guardas passam,
janela após janela, até o processo sair de especificação.

Escreva a **guarda 5b**. Exercício deliberadamente subespecificado — como na vida.

In [ ]:
def guarda_5b(historico_efeitos: list[float], n_janelas: int) -> bool:
    """True = bloquear e encaminhar para manutenção preditiva.

    Deve disparar em RAMPA monotônica (mesmo toda abaixo do limiar).
    Não deve disparar em ruído sem tendência, nem em degrau súbito
    (degrau já é coberto pelas guardas 1 a 5).

    Dica: teste de tendência de Mann-Kendall, ou regressão do efeito
    contra o índice da janela exigindo coeficiente > 0 e p < 0.05.
    """
    # TODO: sua guarda aqui
    raise NotImplementedError

In [ ]:
from driftkit.testing import checar_guarda_5b

checar_guarda_5b(guarda_5b)

In [ ]:
# ============================================================
#  O contrato v2 — a linhagem preservada
# ============================================================
from driftkit.state import Contrato, salvar_contrato

requer("PISO_REAL", "PSI_ALARME_REAL", "cmp", "H")

contrato_v2 = Contrato(
    versao="v2-bosch",
    origem=f"Bosch, semanas de referência {SEM_REF}, recalibrado a partir de {CFG_V1['versao']}",
    features={"num": FEATS_BOSCH, "cat": []},
    limiares={
        "psi_piso_aa": PISO_REAL,
        "psi_alarme": PSI_ALARME_REAL,
        "alpha": 0.05,
        "min_obs": 2000,
        "bins": det_bosch.bins,
        "sigma_carta": 3.0,
        "H_referencia": None if not np.isfinite(H) else round(float(H), 2),
    },
    guardas_retreino=CFG_V1["guardas_retreino"],                   # método: transfere
    assinaturas_diagnosticas=CFG_V1["assinaturas_diagnosticas"],   # método: transfere
    fixture={
        "features_monitoradas": len(FEATS_BOSCH),
        "n_referencia": int(n_ref),
        "amostras_por_janela": int(n_jan),
        "prevalencia": round(float(base.Response.mean()), 5),
        "calibracao": {k: v for k, v in cal_v2.items()},
    },
    suite={"pass": CFG_V1["suite"]["pass"], "total": CFG_V1["suite"]["total"]},
    linhagem={"deriva_de": CFG_V1["versao"], **cmp},
)
salvar_contrato(contrato_v2, DATA / "detector_config_v2.json")
print()
print(contrato_v2.resumo())
print(f"\nlinhagem: v1 → v2")
print(f"  {cmp['veredicto']}")

In [ ]:
checkpoint(
    5,
    politica_decidindo=d.acao is Acao.BLOQUEAR,
    exit_code_correto=d.exit_code == 20,
    guarda_5_e_a_unica=d.guardas.get("5_causa_e_ml") is False,
    escada_medida=np.isfinite(g),
    contrato_v2_salvo=(DATA / "detector_config_v2.json").exists(),
)

---
# 🎓 Bloco VI — Fechamento e Q&A (15 min)

## O que a nossa suíte **não** pode provar

Um monitor honesto sabe dizer onde é cego:

- **Não provamos** que o detector funciona em *qualquer* fábrica. Provamos que
  funciona nos casos que plantamos.
- **Não provamos** que os limiares do Bosch estão certos. Não há ground truth.
  Provamos que são **coerentes com o piso de ruído previsto e medido** — o que é
  bem menos, e é o máximo honestamente disponível.
- **Não cobrimos** deriva lenta multivariada, drift em interações, nem mudança
  de rota simultânea a mudança de valores.

> Isso não é fraqueza do método. É o método funcionando: um detector que
> declara suas limitações é utilizável; um que não declara é apenas silencioso.

---

## Se você esquecer todo o resto, leve três coisas

> **1.** O drift que mais move as distribuições é o que menos machuca.
> O que mais machuca é quase invisível nelas.
>
> **2.** O piso de ruído do seu detector é **previsível**. E o `0.1` que você
> herdou corresponde a uma janela de ~340 observações — não à sua.
>
> **3.** Um pipeline que sabe se **recusar** a retreinar é mais maduro que um
> que retreina rápido.

---

## 🚀 Segunda-feira, 15 minutos

**Divida sua referência pela metade no tempo — não aleatoriamente.**
Compare o PSI com $$\chi^2_{0,95}(B-1)\cdot(2/n)$$.

Se der mais que o dobro, sua janela de referência contém mais de um regime —
e nenhum limiar te salva disso.

In [ ]:
# copie isto para o seu trabalho. É a única célula que importa depois de hoje.
from driftkit.detectors import DriftDetector, piso_analitico, n_equivalente

# sua_referencia = sua_referencia.sort_values("data")   # ⚠️ ORDENE PRIMEIRO
# det = DriftDetector.from_reference(sua_referencia)
# print(det.calibrar_piso(modo="ambos", n_blocos=4))
#
# Leia o `H`:
#   H < 2  → referência homogênea, limiar confiável
#   H >= 5 → referência CONTAMINADA. O problema não é o limiar,
#            é o gabarito. Encurte ou segmente.

print("o limiar 0.1, com 10 bins, corresponde a uma janela de "
      f"~{n_equivalente(0.10):.0f} observações.")
for n in (340, 2_700, 40_000, 500_000):
    p = piso_analitico(n, n)
    print(f"  n={n:>8,}  →  piso {p:.6f}   ·   o 0.1 está {0.10/p:>6.0f}× solto")

print("\n📦 github.com/arcursino/python-br-2026")
print("\nObrigado. Perguntas?")

---
## 🏃 Para levar para casa

| | desafio |
|---|---|
| ⭐ | Rode `calibrar_piso(modo="aleatorio")` e `modo="bloco"` na mesma referência. Explique a diferença para alguém em duas frases. |
| ⭐⭐ | Troque KS por Wasserstein no monitor do Bosch. As features apontadas mudam? |
| ⭐⭐ | Aumente `n_blocos` para 8. O `bloco_mais_divergente` aponta sempre a mesma transição? Se sim, você achou a semana em que algo mudou. |
| ⭐⭐⭐ | Implemente a **guarda 5b** e escreva o teste dela em `tests/`. Abra um PR. |
| ⭐⭐⭐ | Agregue o monitor **por estação** em vez de por feature. Um alerta "a estação L3_S32 driftou" é acionável; "a feature 3939 driftou" não é. |
| 🏆 | Construa o eixo temporal a partir do `Id` e **documente o que quebra**. É o melhor exercício da lista: você vai reproduzir o vazamento da competição e entender por que ele rendeu leaderboard e nenhum aprendizado. |

## Referências

- Gama, J. et al. (2014). *A survey on concept drift adaptation.* ACM Computing Surveys.
- Sculley, D. et al. (2015). *Hidden technical debt in ML systems.* NeurIPS.
- Widmer, G. & Kubat, M. (1996). *Learning in the presence of concept drift.*
- Bifet, A. & Gavaldà, R. (2007). *Learning from time-changing data with adaptive windowing.*
- Yurdakul, B. (2018). *Statistical Properties of Population Stability Index.* Western Michigan University.
- Bosch Production Line Performance — Kaggle, 2016.